#### Forecast API

In [29]:
import openmeteo_requests
import requests_cache
import pandas as pd
import numpy as np
from retry_requests import retry
from pathlib import Path

def main():
    # 1) Forecast meteo
    cache = requests_cache.CachedSession('.cache', expire_after=3600)
    sess  = retry(cache, retries=5, backoff_factor=0.2)
    client = openmeteo_requests.Client(session=sess)

    vars_hr = [  # tu lista completa
        "temperature_2m", "dew_point_2m", "relative_humidity_2m", "apparent_temperature",
        "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "visibility",
        "evapotranspiration", "et0_fao_evapotranspiration", "vapour_pressure_deficit", "wind_speed_10m",
        "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m",
        "wind_direction_120m", "wind_direction_180m", "wind_gusts_10m", "temperature_80m", "temperature_120m",
        "temperature_180m", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm",
        "soil_temperature_54cm", "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm",
        "soil_moisture_9_to_27cm", "soil_moisture_27_to_81cm", "uv_index", "uv_index_clear_sky", "is_day",
        "sunshine_duration", "wet_bulb_temperature_2m", "cape", "lifted_index", "convective_inhibition",
        "freezing_level_height", "boundary_layer_height", "shortwave_radiation", "diffuse_radiation",
        "global_tilted_irradiance", "shortwave_radiation_instant", "diffuse_radiation_instant",
        "global_tilted_irradiance_instant", "direct_radiation", "direct_normal_irradiance",
        "terrestrial_radiation", "direct_radiation_instant", "direct_normal_irradiance_instant",
        "terrestrial_radiation_instant", "pressure_msl"
    ]

    params = {
        "latitude": 18.2158,
        "longitude": -71.0998,
        "hourly": vars_hr,
        "timezone": "UTC",
        "past_days": 1,
        "forecast_days": 7,
        "models": "best_match"
    }

    url = "https://api.open-meteo.com/v1/forecast"
    resps = client.weather_api(url, params=params)
    if not resps:
        raise RuntimeError("No response from forecast API")
    hr = resps[0].Hourly()
    t0 = pd.to_datetime(hr.Time(),      unit="s", utc=True)
    t1 = pd.to_datetime(hr.TimeEnd(),   unit="s", utc=True)
    freq = pd.Timedelta(seconds=hr.Interval())
    idx  = pd.date_range(t0, t1, freq=freq, inclusive="left", tz="UTC")

    df_m = pd.DataFrame(
        {v: hr.Variables(i).ValuesAsNumpy() for i,v in enumerate(vars_hr)},
        index=idx
    )

    # 2) Load histórico de generación
    hist_file = Path("../data/interim/post_despacho_transformed_data/post_despacho_transformed.parquet")
    df_h = pd.read_parquet(hist_file)
    # detecta la columna de generación
    gen_cols = [c for c in df_h.columns if "gen" in c.lower()]
    if not gen_cols:
        raise KeyError(f"No generation column found in {hist_file}")
    gen_col = gen_cols[0]
    df_h = df_h[[gen_col]].rename(columns={gen_col:"generation"})
    # asegurar índice datetime UTC
    if "timestamp" in df_h.columns:
        df_h["timestamp"] = pd.to_datetime(df_h["timestamp"], utc=True)
        df_h = df_h.set_index("timestamp")
    else:
        df_h.index = pd.to_datetime(df_h.index, utc=True)

    # 3) Extraer 24h previas al forecast
    start_fcst = idx.min()
    df_hist24  = df_h.loc[start_fcst - pd.Timedelta(days=1): start_fcst - pd.Timedelta(hours=1)]

    # 4) Construir columna generation: histórico + ceros
    gen = np.zeros(len(df_m), dtype=float)
    # ubica las horas históricas en el índice
    mask = df_m.index.isin(df_hist24.index)
    gen[mask] = df_hist24.reindex(df_m.index[mask]).generation.values
    df_m["generation"] = gen

    # 5) Save combined raw
    out = Path("../data/raw/forecast_meteo_data/parque_solar_girasol_forecast_api_request.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    df_m.reset_index().rename(columns={"index":"date"}).to_csv(out, index=False)
    print("✅ Saved meteo+generation raw to:", out)

if __name__ == "__main__":
    main()


✅ Saved meteo+generation raw to: ..\data\raw\forecast_meteo_data\parque_solar_girasol_forecast_api_request.csv


In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1) Rutas
raw_csv        = Path("../data/raw/forecast_meteo_data/parque_solar_girasol_forecast_api_request.csv")
interim_folder = Path("../data/interim/forecast_meteo_data_transform")
interim_folder.mkdir(parents=True, exist_ok=True)
output_csv     = interim_folder / "parque_solar_girasol_forecast_api_transformed.csv"

# 2) Funciones de feature engineering (idénticas a las del notebook de entrenamiento)
def add_temporal_features(df):
    df = df.copy()
    df["hour"]        = df.index.hour
    df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_of_week"] = df.index.dayofweek
    df["dow_sin"]     = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"]     = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month"]       = df.index.month
    df["month_sin"]   = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"]   = np.cos(2 * np.pi * df["month"] / 12)
    return df

def add_lag_features(df, cols, lags=[1,2,3]):
    df = df.copy()
    for col in cols:
        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df

def add_moving_average_features(df, cols, windows=[3,6]):
    df = df.copy()
    for col in cols:
        for w in windows:
            df[f"{col}_ma{w}"] = df[col].rolling(window=w, min_periods=1).mean()
    return df

# 3) Cargar el CSV raw y fijar índice datetime
df = (
    pd.read_csv(raw_csv, parse_dates=["date"])
      .rename(columns={"date": "timestamp"})
      .set_index("timestamp")
)

# 4) Conservar sólo las variables que entrenó el modelo
weather_vars = [
    "temperature_2m", "dew_point_2m", "relative_humidity_2m", "apparent_temperature",
    "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid",
    "et0_fao_evapotranspiration", "vapour_pressure_deficit", "wind_speed_10m",
    "wind_direction_10m", "wind_gusts_10m", "is_day", "sunshine_duration",
    "wet_bulb_temperature_2m", "boundary_layer_height", "shortwave_radiation",
    "diffuse_radiation", "global_tilted_irradiance", "shortwave_radiation_instant",
    "diffuse_radiation_instant", "global_tilted_irradiance_instant",
    "direct_radiation", "direct_normal_irradiance", "terrestrial_radiation",
    "direct_radiation_instant", "direct_normal_irradiance_instant",
    "terrestrial_radiation_instant", "pressure_msl"
]
df = df[weather_vars]

# 5) Añadir columna 'generation' a cero para poder calcular lags y MA
df["generation"] = 0.0

# 6) Aplicar feature engineering
df = add_temporal_features(df)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df = add_lag_features(df, numeric_cols, lags=[1,2,3])
df = add_moving_average_features(df, numeric_cols, windows=[3,6])

# 7) Eliminar la columna cruda 'generation' para quedarse con 239 features
df = df.drop(columns=["generation"])

# 8) Eliminar filas con NaN generadas por los lags/MA
df_transformed = df.dropna()

# 9) Guardar el CSV transformado
df_transformed.to_csv(output_csv)
print(f"✅ Transformaciones completadas ({df_transformed.shape[1]} columnas) y guardadas en:\n  {output_csv}")

✅ Transformaciones completadas (239 columnas) y guardadas en:
  ..\data\interim\forecast_meteo_data_transform\parque_solar_girasol_forecast_api_transformed.csv


C:\Users\ferna\AppData\Local\Temp\ipykernel_13408\2732237878.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{lag}"] = df[col].shift(lag)
C:\Users\ferna\AppData\Local\Temp\ipykernel_13408\2732237878.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag{lag}"] = df[col].shift(lag)
C:\Users\ferna\AppData\Local\Temp\ipykernel_13408\2732237878.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

In [31]:
import pandas as pd
import joblib
from pathlib import Path

# 1) Path to your transformed‑features CSV (adjust as needed)
TRANSFORMED_CSV = Path("../data/interim/forecast_meteo_data_transform/parque_solar_girasol_forecast_api_transformed.csv")

# 2) Load the features DataFrame
df = pd.read_csv(TRANSFORMED_CSV, index_col="timestamp", parse_dates=["timestamp"])

# 3) Load your trained ensemble model
MODEL_PATH = Path("../data/models/parque_solar_girasol_ensemble.joblib")
model = joblib.load(MODEL_PATH)
expected_feats = list(model.feature_names_in_)

# 4) Reorder to match the model’s feature_names_in_
df = df[expected_feats]

# 5) Predict generation
preds = model.predict(df)

# 6) Build output DataFrame
df_gen = pd.DataFrame(preds, index=df.index, columns=["generation"])

# 7) Save to CSV instead of Parquet
OUT_CSV = Path("../data/processed_predictions/parque_solar_girasol_generation.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_gen.to_csv(OUT_CSV)

print("✅ Saved generation predictions to CSV at:", OUT_CSV)
print(df_gen.head())

✅ Saved generation predictions to CSV at: ..\data\processed_predictions\parque_solar_girasol_generation.csv
                           generation
timestamp                            
2025-04-15 03:00:00+00:00   32.598993
2025-04-15 04:00:00+00:00   32.445604
2025-04-15 05:00:00+00:00   32.445604
2025-04-15 06:00:00+00:00   32.445604
2025-04-15 07:00:00+00:00   32.425419
